# GCP Orphaned Projects Audit

Audits the projects listed in `Google_Orphant_Accounts.txt` to decide which ones can be shut down:

- **Cost** — net cost per month (last 13 months) and the services behind it, from the BigQuery billing export
- **Access** — IAM owners and the users/groups bound to each project
- **Usage** — who last changed or used each project (Cloud Audit Logs), plus the top 5 service accounts still running in it

Output: an HTML report (`GCP_Orphaned_Projects_Audit_<date>.html`) and a CSV with one row per project.

**Before running:** `gcloud auth application-default login` (same credentials as `GCP_FinOps.ipynb`).
The log queries are rate limited (60 reads/min), so a full run takes about 15–20 minutes for ~30 projects.

In [ ]:
import json, re, time
from datetime import date, datetime, timedelta, timezone
from dateutil.relativedelta import relativedelta

import pandas as pd
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery

print("Imports OK")

In [ ]:
# ── Settings ───────────────────────────────────────────────────────────────
ACCOUNTS_FILE   = 'Google_Orphant_Accounts.txt'          # one project name or ID per line
BILLING_PROJECT = 'billing-query-371121'
TEMPLATE_FILE   = 'templates/orphan_audit_template.html'
AS_OF           = date.today()

# 13 invoice months: 12 complete months + the current (partial) month
first_this_month = AS_OF.replace(day=1)
MONTHS     = [(first_this_month - relativedelta(months=i)).strftime('%Y%m') for i in range(12, -1, -1)]
SINCE      = first_this_month - relativedelta(months=13)  # Admin Activity logs are kept 400 days
READS_DAYS = 30                                            # Data Access logs are kept 30 days by default

out_base = f"GCP_Orphaned_Projects_Audit_{AS_OF.isoformat()}"
print(f"As of {AS_OF} | months {MONTHS[0]}–{MONTHS[-1]} | changes since {SINCE}")

## 1. Billing — match names to projects, then pull monthly cost and services

In [ ]:
client = bigquery.Client(project=BILLING_PROJECT)
dataset_id = f"{BILLING_PROJECT}.{list(client.list_datasets())[0].dataset_id}"
tables = list(client.list_tables(dataset_id))
table_id = next(f"{t.project}.{t.dataset_id}.{t.table_id}" for t in tables if "resource" not in t.table_id)
print(f"Using table: {table_id}")

names = [l.strip() for l in open(ACCOUNTS_FILE) if l.strip()]
all_projects = client.query(f"""
SELECT DISTINCT project.id AS pid, project.name AS pname
FROM `{table_id}` WHERE project.id IS NOT NULL
""").to_dataframe()

wanted = {n.lower() for n in names}
projects = all_projects[all_projects.pname.str.lower().isin(wanted) | all_projects.pid.str.lower().isin(wanted)]
matched = set(projects.pname.str.lower()) | set(projects.pid.str.lower())
missing = [n for n in names if n.lower() not in matched]
ids = sorted(projects.pid)
print(f"{len(names)} names → {len(ids)} projects" + (f" | NOT FOUND in billing: {missing}" if missing else ""))

In [ ]:
job = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ArrayQueryParameter('ids', 'STRING', ids),
    bigquery.ScalarQueryParameter('m0', 'STRING', MONTHS[0]),
])
credits = "(SELECT IFNULL(SUM(c.amount), 0) FROM UNNEST(credits) c)"
base = f"FROM `{table_id}` WHERE project.id IN UNNEST(@ids)"

monthly_df = client.query(f"""
SELECT project.id AS pid, invoice.month AS month, SUM(cost) + SUM({credits}) AS net
{base} AND invoice.month >= @m0 GROUP BY 1, 2""", job_config=job).to_dataframe()

services_df = client.query(f"""
SELECT project.id AS pid, service.description AS service, SUM(cost) AS gross, SUM(cost) + SUM({credits}) AS net
{base} AND invoice.month >= @m0 GROUP BY 1, 2""", job_config=job).to_dataframe()

lifetime_df = client.query(f"""
SELECT project.id AS pid,
       MAX(IF(cost > 0.001, usage_end_time, NULL)) AS last_cost,
       MAX(usage_end_time)                         AS last_usage_row,
       MIN(usage_start_time)                       AS first_seen,
       SUM(cost)                                   AS life_gross
{base} GROUP BY 1""", job_config=job).to_dataframe()

print(f"Monthly rows: {len(monthly_df):,} | service rows: {len(services_df):,}")

## 2. Project details, IAM and audit logs

Uses the same Application Default Credentials through the REST APIs. Projects that return 403 are either deleted or not visible to this account.

In [ ]:
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
session = AuthorizedSession(creds)

def call(method, url, body=None):
    """Call a Google API, backing off on rate limits (HTTP 429)."""
    for attempt in range(6):
        r = session.request(method, url, json=body)
        if r.status_code != 429:
            return r
        time.sleep(15 * (attempt + 1))
    return r

# Calls that fire for every project when someone opens the console or billing pages
# (and this audit's own reads). They say nothing about real use, so they are ignored.
NOISE = re.compile(r'(GetResourceBillingInfo|GetProject$|Projects\.GetProject|SearchProjects|GetIamPolicy|'
                   r'ListLogEntries|GetEffectivePolicy|GetEffectiveOrgPolicy|ListEffectiveTags|SearchAllResources|'
                   r'getIamPermissions|TestIamPermissions|ParseTimeSeriesFilter|ListProjects|GetOrganization|'
                   r'ListFolders|GetFolder)', re.I)
HUMAN = 'protoPayload.authenticationInfo.principalEmail:"@" AND NOT protoPayload.authenticationInfo.principalEmail:"gserviceaccount.com"'

def read_logs(pid, flt, pages=3):
    """Newest-first audit log entries for one project. Returns (entries, error_code)."""
    entries, token = [], None
    for _ in range(pages):
        body = {'resourceNames': [f'projects/{pid}'], 'filter': flt, 'orderBy': 'timestamp desc', 'pageSize': 1000}
        if token:
            body['pageToken'] = token
        r = call('POST', 'https://logging.googleapis.com/v2/entries:list', body)
        if not r.ok:
            return entries, r.status_code
        j = r.json()
        for e in j.get('entries', []):
            pp = e.get('protoPayload', {})
            entries.append({'ts': e['timestamp'], 'log': e['logName'].split('%2F')[-1],
                            'who': pp.get('authenticationInfo', {}).get('principalEmail', ''),
                            'svc': pp.get('serviceName', ''), 'method': pp.get('methodName', '')})
        token = j.get('nextPageToken')
        if not token:
            break
        time.sleep(1.1)
    return entries, None

since_ts = f"{SINCE.isoformat()}T00:00:00Z"
reads_ts = (datetime.now(timezone.utc) - timedelta(days=READS_DAYS)).strftime('%Y-%m-%dT%H:%M:%SZ')

gcp = {}
for i, pid in enumerate(ids, 1):
    d = {}
    r = call('GET', f'https://cloudresourcemanager.googleapis.com/v3/projects/{pid}')
    d['meta'] = r.json() if r.ok else {'error': r.status_code}
    r = call('POST', f'https://cloudresourcemanager.googleapis.com/v1/projects/{pid}:getIamPolicy', {})
    d['iam'] = r.json() if r.ok else {'error': r.status_code}
    # Changes by people (Admin Activity, ~13 months)
    d['writes'], d['log_error'] = read_logs(pid, f'logName:"cloudaudit.googleapis.com%2Factivity" AND timestamp>="{since_ts}" AND {HUMAN}')
    # Hands-on reads by people (Data Access, last 30 days)
    d['reads'], _ = read_logs(pid, f'logName:"cloudaudit.googleapis.com%2Fdata_access" AND timestamp>="{reads_ts}" AND {HUMAN}')
    # Service accounts: newest 3,000 entries of any audit log
    d['bots'], _ = read_logs(pid, f'logName:"cloudaudit.googleapis.com" AND protoPayload.authenticationInfo.principalEmail:"gserviceaccount.com"')
    gcp[pid] = d
    print(f"[{i:>2}/{len(ids)}] {pid:<34} {d['meta'].get('state', d['meta'].get('error'))!s:<8} "
          f"changes={len(d['writes']):<5} reads={len(d['reads']):<5} bot-entries={len(d['bots'])}", flush=True)
    time.sleep(1.5)

## 3. Classify each project

| Status | Rule |
|---|---|
| **Spend, no owner activity** | ≥ $1/mo (last 3 complete months) and no people activity — or our account cannot see the project |
| **Idle** | < $1/mo, no changes by a person since `SINCE`, no hands-on reads in the last 30 days |
| **No access** | Project not readable (403/404) and < $1/mo |
| **Review** | Last people activity 90–180 days ago |
| **In use** | People activity in the last 90 days |

In [ ]:
NOW = pd.Timestamp(AS_OF, tz='UTC')
full_months = MONTHS[:-1]

def entry_summary(entries):
    out = {}
    for e in entries:                     # entries are newest first
        u = out.setdefault(e['who'], {'who': e['who'], 'last': e['ts'], 'n': 0, 'what': f"{e['svc']} · {e['method']}"})
        u['n'] += 1
    return out

rows = []
for _, lt in lifetime_df.iterrows():
    pid = lt.pid
    d = gcp.get(pid, {})
    meta, iam = d.get('meta', {}), d.get('iam', {})
    name = projects[projects.pid == pid].pname.iloc[0]
    by_month = monthly_df[monthly_df.pid == pid].set_index('month').net.to_dict()
    series = [round(float(by_month.get(m, 0) or 0), 2) for m in MONTHS]
    cost12, avg3 = sum(series[:-1]), sum(series[-4:-1]) / 3

    svc = services_df[(services_df.pid == pid) & (services_df.gross >= 0.01)].sort_values('gross', ascending=False)
    services = [{'name': s.service, 'net': round(s.net, 2), 'gross': round(s.gross, 2)} for s in svc.itertuples()]

    members = {}
    for b in iam.get('bindings', []):
        for m in b.get('members', []):
            members.setdefault(m, []).append(b['role'].replace('roles/', ''))
    owners = [m for m, roles in members.items() if 'owner' in roles]

    writes = [e for e in d.get('writes', []) if not NOISE.search(e['method'])]
    reads  = [e for e in d.get('reads', [])  if not NOISE.search(e['method']) and e['who'] != 'system@google.com']
    hands_on = sorted(writes + reads, key=lambda e: e['ts'], reverse=True)
    users = sorted(entry_summary(hands_on).values(), key=lambda u: u['last'], reverse=True)
    last_human = users[0]['last'] if users else None
    days_human = (NOW - pd.Timestamp(last_human)).days if last_human else None

    bots = d.get('bots', [])
    top_sa = sorted(entry_summary(bots).values(), key=lambda a: -a['n'])[:5]

    visible = 'bindings' in iam or meta.get('state') == 'ACTIVE'
    last_row = pd.Timestamp(lt.last_usage_row)
    if not visible:
        if avg3 >= 1:
            status, action = 'spend', f'Still billing about ${avg3:,.0f}/mo, but our account cannot read this project. Ask an org admin to identify the owner and confirm it is needed'
        elif (NOW - last_row).days <= 30:
            status, action = 'gone', 'Still exists (billing records this month) but our account has no access. Ask an org admin who owns it, then shut down if unclaimed'
        else:
            status, action = 'gone', f'Likely deleted: no billing records since {last_row.date()}. Confirm in the org project list and drop from tracking'
    elif days_human is None or days_human > 180:
        if avg3 < 1:
            status, action = 'idle', 'Shut down: no people activity and no spend'
        else:
            status, action = 'spend', 'Spending with no people activity: find the owner, then shut down or move to a known team'
    elif days_human > 90:
        status, action = 'review', 'Low recent activity: ask the owner to confirm it is still needed'
    else:
        status, action = 'active', 'In use: confirm the owner and add owner/team labels'

    rows.append(dict(
        name=name, pid=pid, visible=visible, status=status, action=action,
        state=meta.get('state') or ('' if visible else 'Not visible'), created=meta.get('createTime', '')[:10],
        parent=meta.get('parent', ''), labels=meta.get('labels', {}),
        months=series, cost12=round(cost12, 2), avg3=round(avg3, 2), life=round(lt.life_gross, 2),
        last_cost=str(pd.Timestamp(lt.last_cost).date()) if pd.notna(lt.last_cost) else None,
        first_seen=str(lt.first_seen)[:10], services=services,
        owners=owners, members=[{'m': m, 'roles': r} for m, r in sorted(members.items())],
        n_humans_access=sum(m.split(':')[0] in ('user', 'group') for m in members),
        users=users[:25], n_users=len(users), last_human=last_human, days_human=days_human,
        last_write=writes[0] if writes else None, last_any=(bots[:1] or hands_on[:1] or [None])[0],
        top_sa=top_sa, log_window=(bots[-1]['ts'][:10], bots[0]['ts'][:10]) if bots else None,
        recommender=[], iam_err=iam.get('error'), log_error=d.get('log_error'),
    ))

order = {'spend': 0, 'idle': 1, 'gone': 2, 'review': 3, 'active': 4}
rows.sort(key=lambda x: (order[x['status']], -x['cost12']))

summary = pd.DataFrame([{k: r[k] for k in ('name', 'pid', 'status', 'cost12', 'avg3', 'last_human', 'n_users')} for r in rows])
summary

## 4. Recommendations, report and CSV

In [ ]:
mono = lambda s: f'<span class="mono">{s}</span>'
by_status = lambda s: [r for r in rows if r['status'] == s]
steps = []

for r in sorted(by_status('spend'), key=lambda r: -r['avg3'])[:3]:
    top = r['services'][0]['name'] if r['services'] else 'unknown services'
    steps.append(f"<b>Find the owner of {r['name']}</b> ({mono(r['pid'])}). It costs about ${r['avg3']:,.0f}/mo "
                 f"(${r['cost12']:,.0f} in the last 12 months), mostly {top}. {r['action']}.")
if by_status('idle'):
    steps.append(f"<b>Shut down the {len(by_status('idle'))} idle projects</b> after a final check with the IAM owner: "
                 + ', '.join(mono(r['name']) for r in by_status('idle'))
                 + ". Projects that hold shared setup (folders, roles, logging, CI/CD) should be checked with platform engineering first. Shut-down projects stay recoverable for 30 days.")
gone = by_status('gone')
if gone:
    alive = [r for r in gone if r['action'].startswith('Still exists')]
    steps.append(f"<b>Remove the {len(gone)} no-access projects from tracking</b> once deletion is confirmed."
                 + (f" {', '.join(mono(r['name']) for r in alive)} still produce billing records, so they still exist and need an owner." if alive else ''))

# Fastest-growing project: last complete month vs. the first three months
growth = max(rows, key=lambda r: r['months'][-2] - sum(r['months'][:3]) / 3)
if growth['months'][-2] - sum(growth['months'][:3]) / 3 > 100:
    top = growth['services'][0]['name'] if growth['services'] else ''
    steps.append(f"<b>Watch {growth['name']}.</b> It is the fastest-growing cost on the list "
                 f"(${sum(growth['months'][:3]) / 3:,.0f}/mo a year ago, ${growth['months'][-2]:,.0f} last month), mostly {top}. "
                 "Put a budget alert on it.")
steps.append("<b>Fix labels.</b> Ask each team to set real owner, team and cost-center labels (no <span class=\"mono\">placeholder</span>) "
             "so the next audit can name an owner without digging through logs.")

ideas = [
    "Grant the FinOps account org-level read roles (Browser, Security Reviewer, Logs Viewer). The no-access projects would then show owners and activity.",
    "Use Google's <b>Unattended Project Recommender</b>. It flags idle projects automatically, but needs a quota project with the Recommender API enabled for our account.",
    "Run this audit monthly and track how the shut-down list shrinks over time.",
    "Add an org policy or Terraform check that requires owner and team labels on every new project.",
    "Turn on Data Access audit logs for BigQuery and Cloud Storage on the remaining projects, so read-only use is visible too.",
]

# ── HTML report ────────────────────────────────────────────────────────────
mlabel = lambda m: datetime.strptime(m, '%Y%m').strftime('%b %Y')
mshort = lambda m: datetime.strptime(m, '%Y%m').strftime('%b')
fill = {
    '__RANGE__':       f"{mlabel(MONTHS[0])} – {mlabel(MONTHS[-1])}",
    '__PARTIAL__':     f"{datetime.strptime(MONTHS[-1], '%Y%m').strftime('%B %Y')} is partial (through the {AS_OF.day}th).",
    '__AVG__':         f"{mshort(MONTHS[-4])}–{mshort(MONTHS[-2])}",
    '__AVG_FULL__':    f"{mshort(MONTHS[-4])}–{mlabel(MONTHS[-2])}",
    '__12M__':         f"{mlabel(MONTHS[0])} – {mlabel(MONTHS[-2])}",
    '__SINCE_FULL__':  SINCE.strftime('%-d %b %Y'),
    '__SINCE__':       SINCE.strftime('%b %Y'),
    '__BILLING_PROJECT__': BILLING_PROJECT,
}
html = open(TEMPLATE_FILE).read()
for k, v in fill.items():
    html = html.replace(k, v)
data = {'months': MONTHS, 'asof': AS_OF.isoformat(), 'rows': rows, 'steps': steps, 'ideas': ideas}
html = html.replace('__DATA__', json.dumps(data, default=str))
open(f"{out_base}.html", 'w').write(html)

# ── CSV ────────────────────────────────────────────────────────────────────
pd.DataFrame([{
    'Project name': r['name'], 'Project ID': r['pid'], 'State': r['state'], 'Status': r['status'],
    'Recommended action': r['action'],
    f"Net cost last 12 mo ({fill['__12M__']})": r['cost12'], f"Avg monthly ({fill['__AVG_FULL__']})": r['avg3'],
    'Lifetime gross': r['life'],
    **{f'Net {m}': v for m, v in zip(MONTHS, r['months'])},
    'Top services': '; '.join(f"{s['name']} (${s['net']:,.2f})" for s in r['services'][:5]),
    'Last billed cost': r['last_cost'],
    'Last change by a person': f"{r['last_write']['ts'][:10]} {r['last_write']['who']} {r['last_write']['method']}" if r['last_write'] else '',
    'Last hands-on use': (r['last_human'] or '')[:10], 'Days since hands-on use': r['days_human'],
    'People with hands-on use': '; '.join(f"{u['who']} {u['last'][:10]}" for u in r['users'][:10]),
    'Top 5 service accounts (entries, last seen)': '; '.join(f"{a['who']} ({a['n']}, {a['last'][:10]})" for a in r['top_sa']),
    'IAM owners': '; '.join(r['owners']), 'Users/groups with IAM access': r['n_humans_access'],
    'Labels': '; '.join(f'{k}={v}' for k, v in r['labels'].items()), 'Created': r['created'],
} for r in rows]).to_csv(f"{out_base}.csv", index=False)

print(f"Wrote {out_base}.html and {out_base}.csv")
print(summary.status.value_counts().to_string())